# Machine Learning - Programming Assignment
## Comparing Classification models; and deploying an app on Github utilizing Streamlit

**Student Name:** `ADITYA RAJ`  
**Student ID:** `2025AC05657`  
**Date:** `14th August 2026`

In [ ]:
#%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(12)
print('Libraries imported successfully.')

Libraries imported successfully.


## Section 1: Dataset Selection and Loading

**Requirements:**
- ≥500 samples
- ≥12features
- Public dataset (UCI/Kaggle)
- Binary/Multi-Class Classification problem

In [4]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

data = pd.read_csv(url, names=columns, sep=',', skipinitialspace=True)

# Dataset information
dataset_name = "Adult Income"
dataset_source = "UCI ML Repository"
n_samples = len(data)  # Total number of rows
n_features = len(data.columns) - 1  # Number of features (excluding target)
problem_type = "binary_classification"

# Problem statement
problem_statement = """
Predicting whether a person earns more than $50K/year or not based on census data. Result is a boolean value (True if >$50K, False otherwise).
This is useful for socioeconomic analysis and resource allocation.
"""

# Primary evaluation metric
primary_metric = "accuracy"  # not forgertting other metrics like preceision, recall etc., but accuracy is the most straightforward metric for this problem.

# Metric justification
metric_justification = """
Accuracy is chosen here as the class imbalance is mild (~75/25), and it thus provides a straightforward measure of how many predictions are correct out of all predictions made.
In this binary classification problem, it is important to evaluate the overall performance of the model in correctlyclassifying both classes (earning >$50K and <=$50K).
"""

print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Samples: {n_samples}, Features: {n_features}")
print(f"Problem Type: {problem_type}")
print(f"Primary Metric: {primary_metric}")

Dataset: Adult Income
Source: UCI ML Repository
Samples: 32561, Features: 14
Problem Type: binary_classification
Primary Metric: accuracy


## Section 2: Data Preprocessing

In [5]:
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
#From the description on the website: workclass, occupation, native_country have missing values.
print(data['workclass'].unique())
print(data['occupation'].unique())
print(data['native_country'].unique())

<ArrowStringArray>
[       'State-gov', 'Self-emp-not-inc',          'Private',
      'Federal-gov',        'Local-gov',                '?',
     'Self-emp-inc',      'Without-pay',     'Never-worked']
Length: 9, dtype: str
<ArrowStringArray>
[     'Adm-clerical',   'Exec-managerial', 'Handlers-cleaners',
    'Prof-specialty',     'Other-service',             'Sales',
      'Craft-repair',  'Transport-moving',   'Farming-fishing',
 'Machine-op-inspct',      'Tech-support',                 '?',
   'Protective-serv',      'Armed-Forces',   'Priv-house-serv']
Length: 15, dtype: str
<ArrowStringArray>
[             'United-States',                       'Cuba',
                    'Jamaica',                      'India',
                          '?',                     'Mexico',
                      'South',                'Puerto-Rico',
                   'Honduras',                    'England',
                     'Canada',                    'Germany',
                       'Iran'

### Turns out that missing values are represented as '?', so we can replace them with NaN and then drop those rows for simplicity.

In [7]:
data.replace('?', np.nan, inplace=True)
data.dropna(inplace=True)

#also convert the target variable to binary
data['income'] = (data['income'].str.strip() == '>50K').astype(int)

data.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,income
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,1092.007858,88.372489,40.931238,0.248922
std,13.134665,1.056530e+05,2.549995,7406.346497,404.298370,11.979984,0.432396
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [8]:
data.info()

<class 'pandas.DataFrame'>
Index: 30162 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             30162 non-null  int64
 1   workclass       30162 non-null  str  
 2   fnlwgt          30162 non-null  int64
 3   education       30162 non-null  str  
 4   education_num   30162 non-null  int64
 5   marital_status  30162 non-null  str  
 6   occupation      30162 non-null  str  
 7   relationship    30162 non-null  str  
 8   race            30162 non-null  str  
 9   sex             30162 non-null  str  
 10  capital_gain    30162 non-null  int64
 11  capital_loss    30162 non-null  int64
 12  hours_per_week  30162 non-null  int64
 13  native_country  30162 non-null  str  
 14  income          30162 non-null  int64
dtypes: int64(7), str(8)
memory usage: 5.9 MB


In [9]:
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [10]:
#keep a raw (pre-encoding) copy, needed later to build a human-readable test_data.csv for the app.py
raw_data = data.copy()

#dropping education_num as it is a duplicate of education
data.drop('education_num', axis=1, inplace=True)

#categorical columns encoding
categorical_cols = data.select_dtypes(include='str').columns
encoders = {}  # save each fitted encoder so app.py can replicate this exact mapping
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

import joblib
import os
os.makedirs('../saved_models', exist_ok=True)
joblib.dump(encoders, '../saved_models/label_encoders.pkl')

#splitting the data into train and test sets
X = data.drop('income', axis=1)
y = data['income'].values

In [11]:
raw_data

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


In [12]:
data.shape

(30162, 14)

In [13]:
data.head()

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,5,77516,9,4,0,1,4,1,2174,0,40,38,0
1,50,4,83311,9,2,3,0,4,1,0,0,13,38,0
2,38,2,215646,11,0,5,1,4,1,0,0,40,38,0
3,53,2,234721,1,2,5,0,2,1,0,0,40,38,0
4,28,2,338409,9,2,9,5,2,0,0,0,40,4,0


In [14]:
X

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,39,5,77516,9,4,0,1,4,1,2174,0,40,38
1,50,4,83311,9,2,3,0,4,1,0,0,13,38
2,38,2,215646,11,0,5,1,4,1,0,0,40,38
3,53,2,234721,1,2,5,0,2,1,0,0,40,38
4,28,2,338409,9,2,9,5,2,0,0,0,40,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,2,257302,7,2,12,5,4,0,0,0,38,38
32557,40,2,154374,11,2,6,0,4,1,0,0,40,38
32558,58,2,151910,11,6,0,4,4,0,0,0,40,38
32559,22,2,201490,11,4,0,3,4,1,0,0,20,38


In [15]:
y

array([0, 0, 0, ..., 0, 0, 1], shape=(30162,))

In [16]:
# Train-test split — split on indices first so we can trace test rows back to raw_data later
train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=12)

X_train, X_test = X.loc[train_idx].values, X.loc[test_idx].values
y_train, y_test = data.loc[train_idx, 'income'].values, data.loc[test_idx, 'income'].values

In [17]:
# Feature scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../saved_models/scaler.pkl')

['../saved_models/scaler.pkl']

In [18]:
# Data-logging
train_samples = len(X_train_scaled)       # Number of training samples
test_samples = len(X_test_scaled)        # Number of test samples
train_test_ratio = train_samples / (train_samples + test_samples)

print(f"Train samples: {train_samples}")
print(f"Test samples: {test_samples}")
print(f"Split ratio: {train_test_ratio:.1%}")

Train samples: 24129
Test samples: 6033
Split ratio: 80.0%


In [19]:
# Build test_data.csv — raw (pre-encoding), human-readable, using test set indices
# Kept small since Streamlit free tier has limited capacity; only test data, not training data
test_data_raw = raw_data.loc[test_idx]
test_sample = test_data_raw.sample(n=len(test_idx), random_state=12)
test_sample.to_csv('../test_data.csv', index=False)
print(f"Saved test_data.csv with shape: {test_sample.shape}")
test_sample.head()

Saved test_data.csv with shape: (6033, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
29235,45,Self-emp-not-inc,271828,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,20,United-States,0
31562,40,Private,196029,HS-grad,9,Divorced,Transport-moving,Unmarried,White,Male,0,0,45,United-States,0
20174,21,Private,207103,HS-grad,9,Never-married,Handlers-cleaners,Own-child,White,Male,0,0,40,United-States,0
18583,26,Private,258550,Bachelors,13,Never-married,Adm-clerical,Own-child,White,Male,0,0,40,United-States,0
26148,49,Local-gov,159726,11th,7,Divorced,Handlers-cleaners,Unmarried,White,Male,0,0,40,United-States,0


In [20]:
print(categorical_cols)
print(data.dtypes)

Index(['workclass', 'education', 'marital_status', 'occupation',
       'relationship', 'race', 'sex', 'native_country'],
      dtype='str')
age               int64
workclass         int64
fnlwgt            int64
education         int64
marital_status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country    int64
income            int64
dtype: object


## Section 3. Running the models with no hypertuning

In [21]:
#importing the models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                               recall_score, f1_score, matthews_corrcoef,
                               confusion_matrix, classification_report)

#making/checking saved_models directory to save the trained models
os.makedirs('../saved_models', exist_ok=True)

In [22]:
# Define models to train
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=12),
    'Decision Tree': DecisionTreeClassifier(random_state=12),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(random_state=12)
}

results = {}

In [23]:
# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]  # needed for AUC calculation

    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred)
    }
    results[name] = metrics

    # save each trained model
    filename = f"../saved_models/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(model, filename)

    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


Logistic Regression
  Accuracy: 0.7916
  AUC: 0.8039
  Precision: 0.6553
  Recall: 0.3020
  F1: 0.4134
  MCC: 0.3413

Decision Tree
  Accuracy: 0.8134
  AUC: 0.7497
  Precision: 0.6142
  Recall: 0.6251
  F1: 0.6196
  MCC: 0.4960

KNN
  Accuracy: 0.8145
  AUC: 0.8431
  Precision: 0.6338
  Recall: 0.5617
  F1: 0.5956
  MCC: 0.4773

Naive Bayes
  Accuracy: 0.7858
  AUC: 0.8259
  Precision: 0.6308
  Recall: 0.2877
  F1: 0.3951
  MCC: 0.3191

Random Forest
  Accuracy: 0.8571
  AUC: 0.9026
  Precision: 0.7561
  Recall: 0.6087
  F1: 0.6745
  MCC: 0.5899


In [24]:
# to confirm the dataset imbalance, we can print the value counts of the target variable
print(data['income'].value_counts(normalize=True))

income
0    0.751078
1    0.248922
Name: proportion, dtype: float64


### Using GridSearch to find best parameters for models
- Using f1 scoring instead of Accuracy as we just established that the dataset's imbalanced (75/25). - Optimizing for raw accuracy would just make GridSearch want to "predict majority class more," which is the exact problem LR/NB already have. 
- F1 forces it to balance precision and recall.

In [25]:
from sklearn.model_selection import GridSearchCV

# Random Forest tuning
rf_params = {
    'n_estimators': [150, 200, 250, 300, 350],
    'max_depth': [15, 18, 20, 25, 30],
    'min_samples_split': [2, 3, 4]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=12), rf_params, cv=3, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)
print("Best RF params:", rf_grid.best_params_)

# KNN tuning
knn_params = {'n_neighbors': [5, 6, 7, 8, 9]}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=3, scoring='f1', n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)
print("Best KNN params:", knn_grid.best_params_)

# Decision Tree tuning
dt_params = {
    'max_depth': [12, 15, 18, 20, 25],
    'min_samples_split': [8, 10, 12, 15, 20]
}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=12), dt_params, cv=3, scoring='f1', n_jobs=-1)
dt_grid.fit(X_train_scaled, y_train)
print("Best DT params:", dt_grid.best_params_)

Best RF params: {'max_depth': 18, 'min_samples_split': 2, 'n_estimators': 250}
Best KNN params: {'n_neighbors': 7}
Best DT params: {'max_depth': 12, 'min_samples_split': 15}


### Hyperparameter Tuning

- Logistic Regression and Naive Bayes have few/no meaningful hyperparameters to tune for this dataset, so tuning was focused on Decision Tree, KNN, and Random Forest only.
- The models below are retrained using these tuned parameters, overwriting the
baseline (default-hyperparameter) versions saved earlier.

In [26]:
# Grab best models directly from grid search
final_rf = rf_grid.best_estimator_
final_knn = knn_grid.best_estimator_
final_dt = dt_grid.best_estimator_

tuned_models = {
    'Decision Tree': final_dt,
    'KNN': final_knn,
    'Random Forest': final_rf
}

tuned_results = {}  # separate dict, doesn't overwrite baseline `results`

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred)
    }
    tuned_results[name] = metrics

    # new filename, e.g. random_forest_tuned.pkl — doesn't touch the original
    filename = f"../saved_models/{name.replace(' ', '_').lower()}_tuned.pkl"
    joblib.dump(model, filename)

    print(f"\n{name} (tuned)")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


Decision Tree (tuned)
  Accuracy: 0.8440
  AUC: 0.8862
  Precision: 0.7036
  Recall: 0.6196
  F1: 0.6589
  MCC: 0.5602

KNN (tuned)
  Accuracy: 0.8165
  AUC: 0.8556
  Precision: 0.6400
  Recall: 0.5610
  F1: 0.5979
  MCC: 0.4814

Random Forest (tuned)
  Accuracy: 0.8628
  AUC: 0.9119
  Precision: 0.7742
  Recall: 0.6149
  F1: 0.6854
  MCC: 0.6056


In [27]:
import sys
print(sys.executable)

c:\Users\adityA\miniconda3\python.exe


& "c:\Users\adityA\miniconda3\python.exe" -m streamlit run app.py